# Script to run GPT using the batch API

In [1]:
from openai import OpenAI

from scripts.prompting.jup_utils import open_input_file, from_dataset_to_batch_req, upload_batch_request, run_batch, check_batch_status, convert_response

Step-0: Initialize the client

In [11]:
with open('/Users/aloneirew/workspace/DenseEREGraph/data/my_data/api_key.txt', 'r') as file_api:
    api_key = file_api.readline().strip()

_client = OpenAI(api_key=api_key)

_test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/test'
_train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/train'
_dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/EventFull_dev_dot.json')
# _train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/MATRES/in_my_format/train'
# _dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/MATRES_train_dot.json')

_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/density/90per_gpt4o_30exmp.jsonl'
_response_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/density/90per_gpt4o_30exmp.jsonl'
_final_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/density/90per_gpt4o_30exmp_task_description_v2.json'

_num_of_files_to_prepare = -1
_num_of_examples = 1
_selected_file = "30_final.json"
_reduce = 0.1 # reduction of edges in percentage
_model_id = "gpt-4o"

Step-1: Create the request and save it to a file

In [12]:
print("Running Step-1")
from_dataset_to_batch_req(_test_folder, _train_folder, _dot_train_data, _output_file, _num_of_files_to_prepare, _num_of_examples, _selected_file, _model_id, _reduce)

Running Step-1
Using non random file: 30_final.json
Sequentially Selected File: 30_final.json
Number of tokens in prompt=4004
Number of tokens in prompt=4061
Number of tokens in prompt=3999
Number of tokens in prompt=4035
Number of tokens in prompt=4093
Number of tokens in prompt=4033
Number of tokens in prompt=4029
Number of tokens in prompt=3992
Number of tokens in prompt=4042
Number of tokens in prompt=4001
Total number of tokens in all prompts=40289


Step-2: upload the request to the server

In [14]:
print("Running Step-2")
print(_output_file)
_input_request_jsonl = _output_file
_batch_input_file_id = upload_batch_request(_client, _input_request_jsonl)

Running Step-2
/Users/aloneirew/workspace/DenseEREGraph/data/my_data/density/90per_gpt4o_30exmp.jsonl
Batch input file with id file-KhaXgWx24ZbbjixVAc5Vpu uploaded


Step-3: Run the batch

In [15]:
print("Running Step-3")
print(_batch_input_file_id)
_batch_run_id = run_batch(_client, _batch_input_file_id)

Running Step-3
file-KhaXgWx24ZbbjixVAc5Vpu
Batch object created: Batch(id='batch_67728856bbf081908f17cb6bff18ad93', completion_window='24h', created_at=1735559254, endpoint='/v1/chat/completions', input_file_id='file-KhaXgWx24ZbbjixVAc5Vpu', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1735645654, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))


Step-4: Check the status of the batch

In [16]:
print("Running Step-4")
print(_batch_run_id)
check_batch_status(_client, _batch_run_id, _response_file)

Running Step-4
batch_67728856bbf081908f17cb6bff18ad93
Batch status: Batch(id='batch_67728856bbf081908f17cb6bff18ad93', completion_window='24h', created_at=1735559254, endpoint='/v1/chat/completions', input_file_id='file-KhaXgWx24ZbbjixVAc5Vpu', object='batch', status='failed', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=Errors(data=[BatchError(code='token_limit_exceeded', line=None, message='Enqueued token limit reached for gpt-4o in organization org-YMnUn172q6PRLkFPEWWjB609. Limit: 90,000 enqueued tokens. Please try again once some in_progress batches have been completed.', param=None)], object='list'), expired_at=None, expires_at=1735645654, failed_at=1735559255, finalizing_at=None, in_progress_at=None, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))


Step-5: Convert response to DOT format response

In [8]:
print("Running Step-5")
convert_response(_response_file, _final_output_file)
print("Done!")

Running Step-5
Done!


Cancelling a batch

In [13]:
_client.batches.cancel("batch_67710f830ccc8190ba34189e999d8a82")

ConflictError: Error code: 409 - {'error': {'message': "Cannot cancel a batch with status 'failed'.", 'type': 'invalid_request_error', 'param': None, 'code': None}}